# 4. ViT artifacts and registers

Vision transformers can develop tokens with abnormally high norms, mostly in uninformative
background areas (Darcet et al., *Vision Transformers Need Registers*). Registers are extra tokens
added to absorb them. NAVE shows whether these artifacts form concepts of their own and whether they
touch the objects (paper, Section 5.1 and Figure 4).

This notebook needs PyTorch and downloads DINOv2 ViT-S/14 with and without registers from torch hub:

```bash
pip install "nave-xai[torch]"
```

In [ ]:
# %pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

In [ ]:
import torch
from nave_xai.models import TorchBackbone

models = {
    "DINOv2 ViT-S/14": (torch.hub.load("facebookresearch/dinov2", "dinov2_vits14"), 1),
    "DINOv2 ViT-S/14 + registers": (torch.hub.load("facebookresearch/dinov2", "dinov2_vits14_reg"), 5),
}
backbones = {name: TorchBackbone(m, multiple=14, patch_size=14, prefix_tokens=prefix)   # class (+ 4 register) tokens
             for name, (m, prefix) in models.items()}
last = "blocks.11"     # output of the last transformer block (N:12)

### Explanations (K = 5, as in Figure 4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.8))
axes[0].imshow(image); axes[0].set_title("image"); axes[0].axis("off")
exps, explainers = {}, {}
for ax, (name, bb) in zip(axes[1:], backbones.items()):
    explainers[name] = NAVEImage(bb, layers=[last], k=5)
    exps[name] = explainers[name].explain(image)
    show(ax, explainers[name], exps[name], name)

### Which concepts hold the high-norm tokens?

The token norms of the last block, and for every concept the mean norm of its tokens. An artifact
concept stands out with a much higher mean norm; check where it lies in the explanation map.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def token_norms(bb, image):
    h, w = image.shape[0] // 14 * 14, image.shape[1] // 14 * 14
    x = ((image[:h, :w] / 255.0 - IMAGENET_MEAN) / IMAGENET_STD).astype(np.float32).transpose(2, 0, 1)[None]
    feats = bb(x, [last])[last][0]                 # (C, h/14, w/14)
    return np.linalg.norm(feats, axis=0), (h, w)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, (name, bb) in zip(axes, backbones.items()):
    norms, (h, w) = token_norms(bb, image)
    ax.imshow(norms, cmap="magma"); ax.set_title(f"token norms, {name}", fontsize=9); ax.axis("off")
    concepts = exps[name].explanation[7:h:14, 7:w:14][:norms.shape[0], :norms.shape[1]]
    means = [norms[concepts == c].mean() for c in np.unique(concepts)]
    print(name, "mean token norm per concept:", np.round(means, 1))

**Reading the result:** if one concept has a much higher mean norm and sits in the background, the
artifacts are confined there, and the object concepts are not corrupted. Compare its share of the
image with and without registers (`np.bincount(exp.explanation.ravel(), minlength=exp.K)`).

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```

We thank Ulf Brefeld (Leuphana University Lüneburg) for lending us the computing resources without which this work would not have been possible.